# Formula 1 Tyre Degradation Analysis

By Sami Krothapalli

## Step 1: Import Dataset

AeroSpeed F1 Telemetry Car Aero 2022-2024
Dataset Link:  https://www.kaggle.com/datasets/shivmahlan/aerospeed-analytics?utm_source=chatgpt.com


In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

data_folder = "data/F1 Dataset"

os.listdir(data_folder)

laps = pd.read_csv("data/F1 Dataset/f1_race_laps.csv")
laps.head()
print(laps.shape)
laps.columns

(73398, 24)


Index(['Driver', 'DriverNumber', 'Team', 'LapTime', 'LapNumber', 'Compound',
       'TyreLife', 'FreshTyre', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST',
       'Sector1Time', 'Sector2Time', 'Sector3Time', 'PitInTime', 'PitOutTime',
       'TrackStatus', 'IsAccurate', 'GP', 'Location', 'Year', 'Round',
       'HasPitStop'],
      dtype='object')

## Step 2: Clean Dataset
Want to keep only normal dry racing laps

In [14]:
clean_laps = laps.copy()
clean_laps = clean_laps[
    (clean_laps["IsAccurate"] == True) &
    (clean_laps["HasPitStop"] == False) &
    (clean_laps["PitInTime"].isna()) &
    (clean_laps["TrackStatus"] == 1) &
    (clean_laps["Compound"].isin(["SOFT", "MEDIUM", "HARD"])) &
    (clean_laps["LapTime"].notna()) &
    (clean_laps["TyreLife"].notna())
]
print(clean_laps.shape)
clean_laps.head()

(58521, 24)


,Driver,DriverNumber,Team,LapTime,LapNumber,Compound,TyreLife,FreshTyre,SpeedI1,SpeedI2,...,Sector3Time,PitInTime,PitOutTime,TrackStatus,IsAccurate,GP,Location,Year,Round,HasPitStop
1,VER,1,Red Bull Racing,97.880,2.0,SOFT,5.0,False,230.0,252.0,...,24.326,NaN,NaN,1,True,Bahrain Grand Prix,Sakhir,2022,1,False
2,VER,1,Red Bull Racing,98.357,3.0,SOFT,6.0,False,229.0,254.0,...,24.384,NaN,NaN,1,True,Bahrain Grand Prix,Sakhir,2022,1,False
3,VER,1,Red Bull Racing,98.566,4.0,SOFT,7.0,False,231.0,250.0,...,24.550,NaN,NaN,1,True,Bahrain Grand Prix,Sakhir,2022,1,False
4,VER,1,Red Bull Racing,98.877,5.0,SOFT,8.0,False,229.0,256.0,...,24.525,NaN,NaN,1,True,Bahrain Grand Prix,Sakhir,2022,1,False
5,VER,1,Red Bull Racing,98.940,6.0,SOFT,9.0,False,229.0,255.0,...,24.609,NaN,NaN,1,True,Bahrain Grand Prix,Sakhir,2022,1,False


## Step 3: Create Stints 
    Sting -> Continuous Run on Same set of tires

In [15]:
clean_laps = clean_laps.sort_values(
    ["Year", "Round", "Driver", "LapNumber"]
).copy()

clean_laps["NewStint"] = (
    (clean_laps["Compound"] != clean_laps.groupby(["Year", "Round", "Driver"])["Compound"].shift()) |
    (clean_laps["TyreLife"] < clean_laps.groupby(["Year", "Round", "Driver"])["TyreLife"].shift())
)

clean_laps["Stint"] = clean_laps.groupby(["Year", "Round", "Driver"])["NewStint"].cumsum()

clean_laps[["Year", "GP", "Driver", "LapNumber", "Compound", "TyreLife", "Stint", "LapTime"]].head(20)

,Year,GP,Driver,LapNumber,Compound,TyreLife,Stint,LapTime
437,2022,Bahrain Grand Prix,ALB,2.0,SOFT,2.0,1,100.548
438,2022,Bahrain Grand Prix,ALB,3.0,SOFT,3.0,1,100.664
439,2022,Bahrain Grand Prix,ALB,4.0,SOFT,4.0,1,101.126
440,2022,Bahrain Grand Prix,ALB,5.0,SOFT,5.0,1,102.303
441,2022,Bahrain Grand Prix,ALB,6.0,SOFT,6.0,1,101.708
442,2022,Bahrain Grand Prix,ALB,7.0,SOFT,7.0,1,101.561
443,2022,Bahrain Grand Prix,ALB,8.0,SOFT,8.0,1,103.097
444,2022,Bahrain Grand Prix,ALB,9.0,SOFT,9.0,1,102.149
445,2022,Bahrain Grand Prix,ALB,10.0,SOFT,10.0,1,103.500
446,2022,Bahrain Grand Prix,ALB,11.0,SOFT,11.0,1,102.588
